# Reasoning, Planning, and Bounded Control

> **The story.** ReAct made action traces inspectable, but inspectable does not mean bounded. Production agents need the same controls long used in workflow engines: budgets, terminal states, retry rules, and cycle detection.
>
> **Where you are.** Notebook 00 gave OrderFlow typed tools. PO `#7298` now requests more sensors than inventory holds. The naive controller keeps checking the same fact because it has no definition of progress.
>
> **Notation.** $s_t$ is workflow state; $a_t$ is the action at step $t$; $b_s$, $b_t$, and $b_c$ are step, token, and cost budgets; $f(a_t, o_t)$ is a cycle fingerprint.

## 0 - The Challenge

> **The mission:** every fixture must terminate in at most eight steps, repeated-action loops must be detected, and solvable cases must remain at or above 90% completion.

```mermaid
flowchart LR
    A["Unavailable inventory"] --> B["Naive ReAct loop"]
    B --> C["Same action repeats"]
    C --> D["Plan + budgets + cycle check"]
    D --> E["Explicit terminal state"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Setup: import the deterministic OrderFlow runtime -----------------------
from pathlib import Path
import json
import sys


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "learning" / "agentic-ai" / "shared" / "__init__.py").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from the ai-portfolio repository or a descendant directory.")


REPO_ROOT = find_repo_root()
TRACK_DIR = REPO_ROOT / "learning" / "agentic-ai"
if str(TRACK_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_DIR))
from dataclasses import dataclass, field
from typing import Any

from shared import INVENTORY, SupplierServiceDouble, estimate_tokens, request_by_id, stable_hash

shortage_request = request_by_id("PO-7298")
assert INVENTORY[shortage_request["sku"]]["available"] < shortage_request["quantity"]
print("Walking incident:", shortage_request["email"])

## 1 - Failure First: Plausible Reasoning Can Still Loop

![A repeated inventory loop drains budget on the left while a bounded three-step plan, resource limiters, and cycle detector provide terminal exits on the right](../images/ch01-bounded-control-loop.png)

**Predict:** If the observation never changes, how many times will a controller without a terminal rule call inventory: once, until success, or until an external process kills it?

```mermaid
flowchart LR
    S["Need 15 sensors"] --> C["Check inventory"]
    C --> O["Only 9 available"]
    O --> C
    style S fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style O fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Failure first: an observation without a terminal rule ----------------
def unbounded_inventory_loop(request, demonstration_cap=12):
    trace = []
    for step in range(1, demonstration_cap + 1):
        observation = {
            "available": INVENTORY[request["sku"]]["available"],
            "requested": request["quantity"],
        }
        trace.append({"step": step, "action": "check_inventory", "observation": observation})
        if observation["available"] >= observation["requested"]:
            return trace, "ready"
        # No else branch: the controller interprets failure as "try again".
    return trace, "externally_stopped"

naive_trace, naive_terminal = unbounded_inventory_loop(shortage_request)
repeated_calls = sum(event["action"] == "check_inventory" for event in naive_trace)
print(f"Naive terminal state: {naive_terminal}; repeated inventory calls: {repeated_calls}")
assert naive_terminal == "externally_stopped" and repeated_calls == 12
print("Failure observed: rationale did not provide progress or termination.")


## 2 - Plan Before Execute, Then Check Progress

A plan is not hidden chain-of-thought. It is an inspectable task list with preconditions and terminal outcomes. For an inventory shortage, the plan must change tools instead of asking the same question again.

```mermaid
flowchart TD
    P["Plan"] --> I["Inspect inventory"]
    I --> Q{ "Enough stock?" }
    Q -->|"Yes"| F["Price and finish"]
    Q -->|"No"| S["Request supplier quote"]
    S --> A["Needs approval or finish"]
    style P fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Q fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Build an explicit plan and a structural cycle fingerprint ------------
def make_plan(request):
    return [
        {"task": "check_inventory", "status": "pending"},
        {"task": "get_fresh_quote", "status": "pending"},
        {"task": "route_approval", "status": "pending"},
    ]

def fingerprint(action, arguments, observation):
    return stable_hash({"action": action, "arguments": arguments, "observation": observation})

plan = make_plan(shortage_request)
print(json.dumps(plan, indent=2))
assert [task["task"] for task in plan] == ["check_inventory", "get_fresh_quote", "route_approval"]
print("PASS: the next useful action remains visible after inventory fails.")


## 3 - Budgets, Retry Policy, and Terminal States

Budgets turn an open-ended loop into a bounded computation. A retry is valid only when the error is transient and the next attempt changes something, such as elapsed backoff or provider choice.

$$
C = \sum_{t=1}^{n}(c_{model,t} + c_{tool,t}) \le b_c
$$

The controller adds model and tool cost per step and must stop before the total exceeds the declared cost budget.

```mermaid
flowchart LR
    A["Action"] --> B{ "Progress?" }
    B -->|"Yes"| C["Continue within budgets"]
    B -->|"No"| D{ "Same fingerprint?" }
    D -->|"Yes"| E["Terminal: cycle detected"]
    D -->|"No"| F["Retry if transient"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Implement the bounded controller --------------------------------------
@dataclass
class Budget:
    max_steps: int = 8
    max_tokens: int = 600
    max_cost: float = 8.0

@dataclass
class RunResult:
    terminal: str
    trace: list[dict[str, Any]] = field(default_factory=list)
    tokens: int = 0
    cost: float = 0.0


def run_bounded(request, budget=Budget(), supplier=None):
    supplier = supplier or SupplierServiceDouble()
    result = RunResult(terminal="running")
    seen = set()
    plan = make_plan(request)
    for step, task in enumerate(plan, 1):
        if step > budget.max_steps:
            result.terminal = "step_budget_exhausted"
            break
        action = task["task"]
        arguments = {"sku": request["sku"], "quantity": request["quantity"]}
        if action == "check_inventory":
            observation = {"available": INVENTORY[request["sku"]]["available"]}
        elif action == "get_fresh_quote":
            quotes = supplier.quotes(request["sku"])
            observation = min(
                [quote for quote in quotes if quote["trusted"] and quote["age_hours"] <= 48],
                key=lambda quote: quote["unit_price"],
            )
        else:
            total = result.trace[-1]["observation"]["unit_price"] * request["quantity"]
            observation = {"route": "auto" if total <= 5000 else "manager" if total <= 25000 else "finance", "total": total}
        current_fingerprint = fingerprint(action, arguments, observation)
        if current_fingerprint in seen:
            result.terminal = "cycle_detected"
            break
        seen.add(current_fingerprint)
        step_tokens = estimate_tokens(json.dumps({"action": action, "observation": observation}))
        step_cost = 1.0
        if result.tokens + step_tokens > budget.max_tokens or result.cost + step_cost > budget.max_cost:
            result.terminal = "budget_exhausted"
            break
        result.tokens += step_tokens
        result.cost += step_cost
        result.trace.append({"step": step, "action": action, "observation": observation})
        task["status"] = "complete"
    else:
        result.terminal = "completed"
    return result

bounded = run_bounded(shortage_request)
print(json.dumps({"terminal": bounded.terminal, "steps": len(bounded.trace), "tokens": bounded.tokens, "cost": bounded.cost}, indent=2))
assert bounded.terminal == "completed" and len(bounded.trace) <= 8


In [ ]:
# -- Prove cycle detection independently of the happy path ----------------
def detect_repeated_actions(events):
    seen = set()
    for event in events:
        key = stable_hash({"action": event["action"], "observation": event["observation"]})
        if key in seen:
            return True
        seen.add(key)
    return False

assert detect_repeated_actions(naive_trace)
assert not detect_repeated_actions(bounded.trace)
print(f"Cycle detected in naive trace: {detect_repeated_actions(naive_trace)}")
print(f"Cycle detected in bounded trace: {detect_repeated_actions(bounded.trace)}")
print("PASS: repeated state-action pairs stop the controller instead of consuming the budget silently.")


## 4 - Evaluate the Controller, Not Its Rationale

Completion means reaching a valid terminal state on solvable work. It does not mean the narrative sounded confident.

```mermaid
flowchart LR
    F["Fixture suite"] --> R["Run bounded controller"]
    R --> M["Measure success, steps, budget"]
    M --> G{ "Targets met?" }
    G -->|"Yes"| P["Promote controller"]
    G -->|"No"| D["Inspect failed trajectory"]
    style F fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style M fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Setup: import the deterministic OrderFlow runtime -----------------------
from pathlib import Path
import json
import sys


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "learning" / "agentic-ai" / "shared" / "__init__.py").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from the ai-portfolio repository or a descendant directory.")


REPO_ROOT = find_repo_root()
TRACK_DIR = REPO_ROOT / "learning" / "agentic-ai"
if str(TRACK_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_DIR))
from shared import load_purchase_requests

solvable = [request for request in load_purchase_requests() if request["solvable"]]
results = []
for request in solvable:
    try:
        run = run_bounded(request)
        success = run.terminal == "completed"
        steps = len(run.trace)
    except (KeyError, ValueError):
        success = False
        steps = 0
    results.append({"request_id": request["request_id"], "success": success, "steps": steps})

success_rate = sum(row["success"] for row in results) / len(results)
max_steps = max(row["steps"] for row in results)
print(f"Solvable completion: {sum(row['success'] for row in results)}/{len(results)} ({success_rate:.0%})")
print(f"Maximum observed steps: {max_steps}")
assert success_rate >= 0.90 and max_steps <= 8
print("PASS: bounded control preserves at least 90% solvable completion and terminates within eight steps.")

**Your turn:** set `EXERCISE_MAX_STEPS` to `2`. The same task should stop at `step_budget_exhausted`, proving the budget is enforced rather than documented.


In [ ]:
# -- Your turn: change one budget -----------------------------------------
EXERCISE_MAX_STEPS = 8  # CHANGE THIS: try 2
exercise = run_bounded(shortage_request, Budget(max_steps=EXERCISE_MAX_STEPS))
print(f"Terminal state with max_steps={EXERCISE_MAX_STEPS}: {exercise.terminal}")


## Roadmap Checkpoint

| Constraint | Before | After |
|---|---:|---:|
| Repeated inventory calls | 12 before external stop | Cycle detectable from the second repeat |
| Terminal bound | None | At most 8 steps |
| Solvable fixture completion | Not measured | At least 90%, computed above |

```mermaid
flowchart LR
    A["Bounded controller"] --> B["Next failure: context mixes POs"]
    B --> C["Notebook 02: state and memory"]
    style A fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Coverage Ledger

| Tier | Covered here |
|---|---|
| Built and measured | Explicit plan, step/token/cost budgets, cycle detection, terminal states |
| Explained and illustrated | Retry policy and progress checks |
| Named with a reason | Hidden chain-of-thought, excluded because control must be inspectable |

### Key Takeaways

- Reasoning quality cannot substitute for a terminal-state design.
- Retry only transient failures and change something between attempts.
- A structural fingerprint catches exact loops before the budget disappears.
- Evaluate completion and trajectory, not confidence of prose.
